### CARGAR LOS DATOS YA PREPROCESADOS Y FILTRADOS

In [10]:
import pandas as pd
import numpy as np
import pickle
import warnings
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import HyperbandPruner

import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, log_loss, matthews_corrcoef,
    precision_recall_curve, auc
)

### Carga del Baseline

Para este notebook partimos de los datos preprocesados del baseline original, en lugar de utilizar el dataset filtrado de la [Práctica 1](https://github.com/lolamaturana/credit-risk-prediction).

La preparación de estos datos sigue la estructura del [repositorio de referencia del proyecto](https://github.com/mmartinb75/modelizacion_datos_2026). Así, las mejoras obtenidas podrán atribuirse al modelado aplicado y no a modificaciones adicionales en las variables de entrada.

In [11]:
# Carga de los conjuntos de entrenamiento y test preprocesados
with open('data/filtered/X_train_filtered.pkl', 'rb') as f:
    X_train_final = pickle.load(f)
with open('data/filtered/y_train_filtered.pkl', 'rb') as f:
    y_train = pickle.load(f)
with open('data/filtered/X_test_filtered.pkl', 'rb') as f:
    X_test_final = pickle.load(f)
with open('data/filtered/y_test_filtered.pkl', 'rb') as f:
    y_test = pickle.load(f)

# 1. OPTUNA: a hyperparameter optimization framework

En esta fase sustituimos la optimización manual/AUC de la Práctica 1 por una búsqueda bayesiana utilizando **Optuna (TPESampler)**. El objetivo principal es encontrar un modelo que no solo clasifique bien (discriminación), sino que sus probabilidades emitidas sean fiables (calibración). 

Para cumplir con la rúbrica, he implementado las siguientes variaciones respecto al notebook de referencia (11):

**A. Elección de la Métrica Objetivo: Log Loss**
> **Justificación:** Se ha elegido **Log Loss (Cross-Entropy)** como métrica de optimización. El Log Loss es una *strictly proper scoring rule* que se descompone matemáticamente en *resolución* (discriminación) y *fiabilidad* (calibración). A diferencia del AUC (que solo evalúa el ranking general), el Log Loss penaliza fuertemente las predicciones con alta confianza pero incorrectas. Es la métrica más natural para optimizar árboles de decisión cuando el objetivo es usar las probabilidades directas.

**B. Cambio de Pruner: HyperbandPruner**
> **Justificación:** Se sustituye el `MedianPruner` por el **`HyperbandPruner`**. Hyperband utiliza el algoritmo de *Successive Halving*: en lugar de solo cortar basándose en la mediana, asigna dinámicamente más presupuesto computacional (iteraciones) a las configuraciones más prometedoras y descarta agresivamente las peores desde etapas muy tempranas. Permite explorar un espacio mayor en el mismo tiempo.

**C. Ampliación del Espacio de Búsqueda**
> **Justificación:** Hemos introducido nuevos hiperparámetros para mejorar la regularización:
> - En **LightGBM**, se ha añadido `feature_fraction_bynode` (aleatorización de variables en cada nodo, excelente para prevenir el sobreajuste local).
> - En **XGBoost**, se ha incorporado `gamma` (reducción de pérdida mínima requerida para particionar) y `max_delta_step` (crucial para controlar la actualización de pesos en datasets desbalanceados).

Calibracion (variacion al notebook 10)

Medida de incertidumbre y derivacion a un agente

Persistencia del modelo